# pLTC temporal classification
Run after evaluating the paper configurations. Tables and figures are written to `figures/`.

In [ ]:
from pathlib import Path
import json
import shlex
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import torch

from models.primitives.gated_rc import GatedRCBranch
from utils.checkpoint import load_completed_checkpoint

root = Path(".")
out = Path("figures")
out.mkdir(exist_ok=True)
# Keep published capitalization in the figures. The command list specifies the actual runs.
titles = {x.lower(): x for x in ("FreezerSmallTrain", "MoteStrain", "TwoLeadECG", "PowerCons",
                                 "CBF", "ItalyPowerDemand", "BME", "Slope")}
names, candidates, expected = {}, set(), set()
for line in Path("jobs/train.txt").read_text().splitlines():
    args = shlex.split(line)
    filename = args[args.index("--dataset-name") + 1]
    slug = Path(filename).stem.removeprefix("Dataset_")
    names[filename] = titles.get(slug, slug)
    label = args[args.index("--sweep-config") + 1]
    expected.add((names[filename], label, int(args[args.index("--SEED") + 1])))
    if "--gated-config" in args:
        branch = json.loads(args[args.index("--gated-config") + 1])
        if branch.get("transistor_enabled", True) and "constant_gate_input" not in branch:
            candidates.add(label)
DATASET_NAMES = list(names.values())
methods = ["pRNN-1", "pRNN-2", "pLTC"]
rows = []
paths = sorted(root.glob("*/eval/eval-*/result.txt"))
if not paths:
    # Archived releases grouped all configurations under train/ and eval/.
    paths = sorted((root / "eval").glob("*/eval-*/result.txt"))
for path in paths:
    result = json.loads(path.read_text(encoding="utf-8"))
    cfg = result["training_config"]
    group = path.parents[1]
    project = group.parent if group.name == "eval" else root / "train" / group.name
    if (names.get(cfg["dataset_name"]), cfg["sweep_config"], cfg["SEED"]) not in expected:
        continue
    assert result["status"] == "completed", path
    counts = result["device_inventory"]["counts"]
    rows.append(dict(dataset=names[cfg["dataset_name"]], label=cfg["sweep_config"],
                     seed=cfg["SEED"], valid=result["validation_metrics"]["accuracy"],
                     loss=result["validation_metrics"]["loss"], test=100*result["metrics"]["accuracy"],
                     R=counts["resistor"], T=counts["transistor"], C=counts["capacitor"], path=path,
                     model=project/"models"/("pNN_"+result["training_uuid"])))
runs = pd.DataFrame(rows)
if len(rows) != len(expected) or set(zip(runs.dataset, runs.label, runs.seed)) != expected:
    raise ValueError(f"Expected {len(expected)} distinct evaluations. Found {len(rows)} records.")
best = (runs.sort_values(["valid","loss","seed"], ascending=[False,True,True])
        .drop_duplicates(["dataset","label"]))
full = (best[best.label.isin(candidates)].sort_values(["valid","loss","label"], ascending=[False,True,True])
        .drop_duplicates("dataset"))
selected = pd.concat([best[best.label.eq("prnn_ref")].assign(method=methods[0]),
                      best[best.label.eq("prnn2_ref")].assign(method=methods[1]),
                      full.assign(method=methods[2])])
selected.to_csv(out/"selected_models.csv",index=False)
accuracy = selected.pivot(index="dataset", columns="method", values=["test","C"]).reindex(DATASET_NAMES)
accuracy.loc["Macro average"] = accuracy.mean()
accuracy.to_csv(out/"accuracy_capacitors.csv")
display(accuracy.round(2))

In [ ]:
ablations = ["pltc_learnable_tau","pltc_constgate_learnable_tau","pltc_rconly_fixed","pltc_rconly_learnable_tau"]
table = best[best.label.isin(ablations)].pivot(index="dataset", columns="label", values="test")
table = table.reindex(index=DATASET_NAMES,columns=ablations)
table.insert(0, "Full pLTC", full.set_index("dataset").test)
table.loc["Macro average"] = table.mean()
table.to_csv(out/"gate_ablation.csv")
display(table.round(2))

## Trainability
Use all seeds of each validation-selected configuration.

In [ ]:
distribution = runs.merge(selected[["dataset","label","method"]], on=["dataset","label"], validate="many_to_one")
distribution.groupby(["dataset","method"]).test.agg(["mean","std"]).to_csv(out/"trainability.csv")
fig, axes = plt.subplots(2,4,figsize=(8,4.4),sharey=True,constrained_layout=True)
rng = np.random.default_rng(0)
for dataset, ax in zip(DATASET_NAMES,axes.flat):
    for i,method in enumerate(methods,1):
        values = distribution.loc[(distribution.dataset==dataset)&(distribution.method==method),"test"]
        ax.boxplot(values,positions=[i],widths=.5,showfliers=False)
        ax.scatter(i+rng.uniform(-.14,.14,len(values)),values,s=5,alpha=.65,color=["#0072B2","#56B4E9","#D55E00"][i-1])
    ax.set_title(dataset,fontsize=9)
    ax.set_xticks([1,2,3],methods,rotation=30,fontsize=8)
    ax.set_ylim(0,103)
for ax in axes[:,0]: ax.set_ylabel("Test accuracy (%)")
fig.savefig(out/"trainability.pdf",bbox_inches="tight")
plt.show()

## Gate dynamics
Compare fast and slow passive branches with a prescribed gate pulse, then plot the first MoteStrain test example from the validation-selected pLTC checkpoint.

In [ ]:
row = full.set_index("dataset").loc["MoteStrain"]
result = json.loads(row.path.read_text(encoding="utf-8"))
spec = result["gated_config"]
trace = result["trace"]["batches"][0]
i = 2
t_ms = 1000*np.asarray(trace["time"])
state = np.asarray(trace["state"])[0,0,:,i]
def channel(key):
    return np.asarray(trace[key])[0,0,:,i]
if spec.get("learnable_resistance"):
    lo, hi = spec["resistance_bounds"]
    rho = load_completed_checkpoint(row.model)["model_state"]["cell.branch.log_resistance"].flatten()[i].double()
    resistance = math.exp(math.log(lo)+math.log(hi/lo)*rho.sigmoid().item())
else:
    resistance = spec["resistance"]
base_ms = 1000*resistance*spec["capacitance"]
dt = .002/3
t = np.arange(193)*dt
pulse = (t[:-1]>=.020)&(t[:-1]<.040)
target, gate = np.where(pulse,.7,.1), np.where(pulse,.9,0.)
branch = GatedRCBranch(assume_reciprocal=True,resistance=1e7,capacitance=1e-8)
h = torch.tensor([.1],dtype=torch.float64)
states, recorded = [.1], []
with torch.no_grad():
    for u,g in zip(target,gate):
        h = branch(h,torch.tensor([u]),torch.tensor([g]),dt,trace=recorded.append)
        states.append(h.item())
fig, ax = plt.subplots(3,2,figsize=(6.4,5.4),sharex="col",constrained_layout=True)
def step(axis,times,values,label):
    axis.step(times,np.r_[values,values[-1]],where="post",label=label)
step(ax[0,0],t*1000,target,"Target")
step(ax[0,0],t*1000,gate,"Gate")
for tau,name in [(.001,"Fast RC"),(.1,"Slow RC")]:
    passive = [.1]
    for u in target: passive.append(passive[-1]-np.expm1(-dt/tau)*(u-passive[-1]))
    ax[1,0].plot(t*1000,passive,ls="--",label=name)
    ax[2,0].axhline(1000*tau,ls="--",label=name)
ax[1,0].plot(t*1000,states,label="Gated RC")
step(ax[2,0],t*1000,np.array([r["tau"].item() for r in recorded])*1000,"Gated RC")
step(ax[0,1],t_ms,np.repeat(np.array(trace["input"])[0,:,0],spec["substeps"]),"Input")
step(ax[0,1],t_ms,channel("gate"),"Gate")
step(ax[1,1],t_ms,channel("target"),"Target")
ax[1,1].plot(t_ms,state,label="State")
step(ax[2,1],t_ms,1000*channel("tau"),"Local")
ax[2,1].axhline(base_ms,ls=":",label="Base RC")
for axis in ax[2]: axis.set_yscale("log"); axis.set_xlabel("Time (ms)")
for axis,label in zip(ax[:,0],["Drive / gate (V)","State (V)","Local time scale (ms)"]): axis.set_ylabel(label)
for axis in ax.flat: axis.legend(fontsize=7)
ax[0,0].set_title("Controlled branch")
ax[0,1].set_title("MoteStrain, state 2")
fig.savefig(out/"gate_dynamics.pdf",bbox_inches="tight")
plt.show()